In [1]:
# Install spaCy
!pip install spacy
# Download the English medium model
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 49.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
nlp=spacy.load("en_core_web_md")

In [3]:
ner_labels=nlp.get_pipe('ner').labels
print(ner_labels)

('CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART')


In [4]:
texts = [
    "John goes for a walk in Berlin",
    "Mike is going to the store",
    "Elon Musk is the CEO at Twitter",
    "Bob Smith is the guy behind XYZ-Soft Inc.",
    "Florian Dedov is the guy behind NeuralNine"
]

In [5]:
categories=['ORG','PERSON','LOC']

In [6]:
docs =[nlp(text) for text in texts]

In [7]:
print(docs)

[John goes for a walk in Berlin, Mike is going to the store, Elon Musk is the CEO at Twitter, Bob Smith is the guy behind XYZ-Soft Inc., Florian Dedov is the guy behind NeuralNine]


In [8]:
for doc in docs:
  entities=[]
  for ent in doc.ents:
    if ent.label_ in categories:
      entities.append((ent.text,ent.label_))
  print(entities)


[('John', 'PERSON')]
[('Mike', 'PERSON')]
[('Elon Musk', 'PERSON')]
[('Bob Smith', 'PERSON'), ('XYZ-Soft Inc.', 'ORG')]
[('Florian Dedov', 'PERSON'), ('NeuralNine', 'ORG')]


In [9]:
for doc in docs:
  entities=[]
  for ent in doc.ents:

      entities.append((ent.text,ent.label_))
  print(entities)


[('John', 'PERSON'), ('Berlin', 'GPE')]
[('Mike', 'PERSON')]
[('Elon Musk', 'PERSON')]
[('Bob Smith', 'PERSON'), ('XYZ-Soft Inc.', 'ORG')]
[('Florian Dedov', 'PERSON'), ('NeuralNine', 'ORG')]


In [10]:
texts = [
    'What is the price of 4 bananas?',
    'How much are 16 chairs?',
    'Give me the value of 5 Laptops.',
    'Give me the value of five laptops.'
]


docs = [nlp(text) for text in texts]

for doc in docs:
    entities = []
    for ent in doc.ents:

        entities.append((ent.text, ent.label_))
    print(entities)

[('4', 'CARDINAL')]
[('16', 'CARDINAL')]
[('5', 'CARDINAL')]
[('five', 'CARDINAL')]


# **Fine Tuning With Spacy**

In [11]:
!pip install spacy-lookups-data
import random
import spacy
from spacy.util import minibatch
from spacy.training.example import Example

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 MB 7.3 MB/s eta 0:00:00


In [12]:
# Load the model
nlp = spacy.load('en_core_web_md')

In [14]:
# Define training data
train_data = [
    ("What is the price of 10 bananas?", {"entities": [(21, 23, "QUANTITY"), (24, 31, "PRODUCT")]}),
    ("What is the price of 5 laptops?", {"entities": [(21, 22, "QUANTITY"), (23, 30, "PRODUCT")]}),
    ("How much are 7 bottles?", {"entities": [(13, 14, "QUANTITY"), (15, 22, "PRODUCT")]}),
    ("Could I buy 17 phones from you?", {"entities": [(12, 14, "QUANTITY"), (15, 21, "PRODUCT")]}),
    ("I am interested in acquiring 10 books.", {"entities": [(31, 33, "QUANTITY"), (34, 39, "PRODUCT")]}),
    ("Can you get me 12 apples?", {"entities": [(16, 18, "QUANTITY"), (19, 25, "PRODUCT")]}),
    ("Please check the price of 3 pens.", {"entities": [(26, 27, "QUANTITY"), (28, 32, "PRODUCT")]}),
    ("What do 5 chairs cost?", {"entities": [(8, 9, "QUANTITY"), (10, 16, "PRODUCT")]}),
]


In [15]:
# Access the NER component
if 'ner' not in nlp.pipe_names:
    ner = nlp.add_pipe('ner')
else:
    ner = nlp.get_pipe('ner')

# Add labels
for _, annotations in train_data:
    for ent in annotations['entities']:
        if ent[2] not in ner.labels:
            ner.add_label(ent[2])

In [16]:
# Disable other pipes during training
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']

In [18]:
# Training loop
with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.create_optimizer()
    epochs = 50
    for epoch in range(epochs):
        random.shuffle(train_data)
        losses = {}
        batches = minibatch(train_data, size=2)
        for batch in batches:
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                example = Example.from_dict(doc, annotations)
                examples.append(example)
            nlp.update(examples, drop=0.5, losses=losses)
        print(f"Epoch {epoch+1}, Losses: {losses}")


Epoch 1, Losses: {'ner': np.float32(5.9725676e-07)}
Epoch 2, Losses: {'ner': np.float32(1.5636495e-05)}
Epoch 3, Losses: {'ner': np.float32(0.07729421)}
Epoch 4, Losses: {'ner': np.float32(0.98800063)}
Epoch 5, Losses: {'ner': np.float32(0.00071853027)}
Epoch 6, Losses: {'ner': np.float32(0.00017598264)}
Epoch 7, Losses: {'ner': np.float32(0.00040705493)}
Epoch 8, Losses: {'ner': np.float32(0.004522016)}
Epoch 9, Losses: {'ner': np.float32(6.601979e-06)}
Epoch 10, Losses: {'ner': np.float32(3.0600752e-06)}
Epoch 11, Losses: {'ner': np.float32(2.092306e-06)}
Epoch 12, Losses: {'ner': np.float32(2.745234e-06)}
Epoch 13, Losses: {'ner': np.float32(6.0412304e-07)}
Epoch 14, Losses: {'ner': np.float32(3.4959276e-06)}
Epoch 15, Losses: {'ner': np.float32(8.3619256e-07)}
Epoch 16, Losses: {'ner': np.float32(0.08703222)}
Epoch 17, Losses: {'ner': np.float32(4.8063016e-06)}
Epoch 18, Losses: {'ner': np.float32(1.6481437e-06)}
Epoch 19, Losses: {'ner': np.float32(1.744831e-07)}
Epoch 20, Losses:

In [19]:
# Save the model
nlp.to_disk('custom_ner_model')

In [20]:
# Load the trained model
trained_nlp = spacy.load('custom_ner_model')

In [21]:
test_texts = [
    "How much for 3 oranges?",
    "I want 15 chairs for the conference.",
    "Can you give me the price for 6 desks?"
]

In [22]:
for test in test_texts:
  doc=trained_nlp(text)
  print(f'Text:{text}')
  print('Entities',[(ent.text,ent.label_) for ent in doc.ents])

Text:Please check the price of 3 pens.
Entities [('3', 'QUANTITY'), ('pens', 'PRODUCT')]
Text:Please check the price of 3 pens.
Entities [('3', 'QUANTITY'), ('pens', 'PRODUCT')]
Text:Please check the price of 3 pens.
Entities [('3', 'QUANTITY'), ('pens', 'PRODUCT')]
